# Microsoft Agent Framework — Azure OpenAI (Responses API)

În acest exemplu de cod, veți folosi **Microsoft Agent Framework (MAF)** pentru a crea un agent simplu susținut de **Azure OpenAI** folosind **Responses API**.

> **Notă de migrare:** Acest exemplu folosea anterior Semantic Kernel cu GitHub Models. A fost migrat la Microsoft Agent Framework, iar GitHub Models (învechit, retras în iulie 2026) a fost înlocuit cu Azure OpenAI, care acceptă Responses API. `OpenAIChatClient` în MAF țintește endpointul stabil `/openai/v1/` al Azure OpenAI și folosește implicit Responses API.

Scopul acestui exemplu este să demonstreze pașii care vor fi aplicați ulterior în alte exemple de cod la implementarea diverselor modele agentice.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importați pachetele Python necesare


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definirea unui Instrument

În cadrul Microsoft Agent Framework, un **instrument** este o funcție Python simplă decorată cu `@tool` pe care agentul o poate apela. Mai jos definim un instrument care returnează o destinație de vacanță aleatorie și evită să repete pe cea precedentă.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Crearea Agentului

Aici, creăm Agentul numit `TravelAgent`.

În acest exemplu, folosim instrucțiuni foarte simple. Simțiți-vă liber să modificați aceste instrucțiuni pentru a observa cum se schimbă comportamentul agentului.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Rularea agentului

Acum putem rula agentul. Creăm o `AgentSession` astfel încât agentul să-și amintească conversația pe parcursul tururilor, apoi trimitem două `user_inputs`. Primul cere o excursie; al doilea spune că utilizatorului nu i-a plăcut sugestia și cere alta — agentul folosește istoricul sesiunii plus instrumentul `get_random_destination` pentru a răspunde.

Puteți modifica aceste mesaje pentru a observa cum reacționează agentul diferit. Răspunsurile sunt **transmise în flux** token cu token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
